In [ ]:
# 06 — Keep the FAQ vector search endpoint alive
# The workspace deletes personal Vector Search endpoints after 6 hours without activity.
# This notebook runs every 4 hours (maintops_vs_keepalive job): one small query keeps the endpoint
# active; if the endpoint or index has been deleted anyway, it starts the maintops_rag job to rebuild them.

In [ ]:
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("rag_job_id", "")
RAG_JOB_ID = int(dbutils.widgets.get("rag_job_id"))
INDEX_NAME = "bootcamp_students.maintops.faq_index"

w = WorkspaceClient()

In [ ]:
try:
    res = w.api_client.do("POST", f"/api/2.0/vector-search/indexes/{INDEX_NAME}/query", body={
        "columns": ["chunk_id"],
        "query_text": "How does MaintOps work?",
        "num_results": 1,
    })
    print(f"Index answered ({len(res['result']['data_array'])} row) — endpoint kept alive.")
except Exception as e:
    print(f"Index query failed: {e}")
    active = list(w.jobs.list_runs(job_id=RAG_JOB_ID, active_only=True))
    if active:
        print(f"maintops_rag is already running (run {active[0].run_id}) — not starting another rebuild.")
    else:
        run = w.jobs.run_now(job_id=RAG_JOB_ID)
        print(f"Started maintops_rag run {run.run_id} to rebuild the endpoint and index.")